# Flow Matching + GRPO: Steering Generative Models with Reinforcement Learning

In this notebook you will:

1. **Train a flow matching model** that maps standard Gaussian noise to a 2D *checkerboard* distribution.
2. **Reframe sampling as a sequential decision process** by turning the deterministic ODE into a stochastic SDE.
3. **Fine-tune the model with GRPO** (Group Relative Policy Optimization) to steer the output distribution toward a reward signal.
4. **Study the trade-off** between reward maximization and staying close to the original prior, via KL regularization.

This is a miniature version of what is done at scale for diffusion/flow-based image and video models (DDPO, DPOK), for robotics policies (π0), and structurally for LLM RLHF/RLVR pipelines (PPO, GRPO, DeepSeek-R1).

**Runtime:** the notebook is designed to run end-to-end in ~5 minutes on a Colab GPU, or ~15 minutes on CPU. All compute-heavy cells print progress.

---

### Recap: what we already know

You have already seen PPO, so the RL machinery — policy gradients, importance sampling ratios, clipped objective, advantages — is familiar. What is **new** here is:

- The "policy" is a *flow matching* model, not a categorical head over actions.
- The "trajectory" is the integration path of an ODE turned into an SDE.
- We use **GRPO**, which replaces PPO's learned value function with a *group-relative baseline*: sample $N$ trajectories per condition, compute advantages by normalizing rewards within that group.

If you have not seen flow matching, the next section will cover what you need.


## 0. Setup

In [ ]:
# Standard imports. PyTorch is the only ML dependency.
import copy
import math
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

torch.manual_seed(0)
np.random.seed(0)

## 1. Flow matching primer

### The idea in one paragraph

Flow matching trains a neural network $v_\theta(x, t)$ — a *velocity field* — such that integrating the ODE
$$\frac{dx}{dt} = v_\theta(x, t), \quad t \in [0, 1]$$
starting from $x_0 \sim p_0$ (a simple distribution, e.g. $\mathcal{N}(0, I)$) produces $x_1 \sim p_1$ (the target distribution, here the checkerboard).

### The trick: conditional flow matching

A naive objective ("regress against the true marginal velocity") is intractable, because we do not have access to the marginal velocity field. The **conditional flow matching** insight is that we can regress against a *conditional* velocity that has the same gradient in expectation.

For the simplest linear path between samples $x_0 \sim p_0$ and $x_1 \sim p_1$:

$$x_t = (1 - t)\, x_0 + t\, x_1$$

the conditional velocity is just the constant $u(x_t \mid x_0, x_1) = x_1 - x_0$. The training objective becomes:

$$\mathcal{L}_{\text{CFM}}(\theta) = \mathbb{E}_{t \sim U[0,1],\, x_0 \sim p_0,\, x_1 \sim p_1} \left[ \|v_\theta(x_t, t) - (x_1 - x_0)\|^2 \right]$$

That is *the entire training loop*. No score function, no noise schedule, no ELBO.

### Why this is so much simpler than diffusion

Diffusion model training requires defining a noise schedule, deriving a score-matching objective, and handling numerical issues at $t \to 0$. Flow matching just says: "pick a straight line between noise and data, regress the velocity along it." The connection to diffusion is deep (flow matching subsumes a class of diffusion models when the path is chosen appropriately), but the implementation is dramatically cleaner.

### 1.1 The target distribution

In [ ]:
def sample_checkerboard(n: int) -> torch.Tensor:
    """Sample n points from a checkerboard in [-2, 2]^2.

    8 unit-square modes arranged in a checkerboard pattern.
    """
    x1 = torch.rand(n) * 4 - 2                    # x in [-2, 2)
    x2 = torch.rand(n) - torch.randint(0, 2, (n,)).float() * 2
    x2 = x2 + (torch.floor(x1) % 2)               # shift to create checker pattern
    return torch.stack([x1, x2], dim=1)


# Visualize
pts = sample_checkerboard(4000)
fig, ax = plt.subplots(figsize=(5, 5))
ax.scatter(pts[:, 0], pts[:, 1], s=2, alpha=0.5)
ax.set_xlim(-2.5, 2.5); ax.set_ylim(-2.5, 2.5)
ax.set_aspect("equal")
ax.set_title("Target: 2D checkerboard distribution")
plt.show()

### 1.2 The velocity network

In [ ]:
def sinusoidal_embedding(t: torch.Tensor, dim: int = 32) -> torch.Tensor:
    """Sinusoidal embedding for the scalar time variable t in [0, 1]."""
    half = dim // 2
    freqs = torch.exp(-math.log(10000) * torch.arange(half, device=t.device) / half)
    args = t.view(-1, 1) * freqs.view(1, -1) * 2 * math.pi
    return torch.cat([torch.sin(args), torch.cos(args)], dim=-1)


class VelocityNet(nn.Module):
    """v_theta(x, t): a small MLP that outputs a 2D velocity vector."""

    def __init__(self, hidden: int = 256, t_emb_dim: int = 32):
        super().__init__()
        self.t_emb_dim = t_emb_dim
        in_dim = 2 + t_emb_dim
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden), nn.SiLU(),
            nn.Linear(hidden, hidden), nn.SiLU(),
            nn.Linear(hidden, hidden), nn.SiLU(),
            nn.Linear(hidden, hidden), nn.SiLU(),
            nn.Linear(hidden, 2),
        )

    def forward(self, x: torch.Tensor, t: torch.Tensor) -> torch.Tensor:
        t_emb = sinusoidal_embedding(t, self.t_emb_dim)
        return self.net(torch.cat([x, t_emb], dim=-1))

### 1.3 Training the flow

In [ ]:
def train_flow_matching(steps: int = 4000, batch: int = 1024, lr: float = 1e-3):
    model = VelocityNet().to(device)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=steps)

    losses = []
    for step in range(steps):
        x1 = sample_checkerboard(batch).to(device)       # data
        x0 = torch.randn_like(x1)                         # noise
        t = torch.rand(batch, device=device)
        # Linear interpolation path
        xt = (1 - t).view(-1, 1) * x0 + t.view(-1, 1) * x1
        target = x1 - x0                                  # conditional velocity
        pred = model(xt, t)
        loss = ((pred - target) ** 2).mean()

        opt.zero_grad()
        loss.backward()
        opt.step()
        sched.step()
        losses.append(loss.item())

        if step % 500 == 0:
            print(f"  step {step}: loss {loss.item():.4f}")
    return model, losses


print("Training base flow matching model...")
base_model, losses = train_flow_matching(steps=4000)
print("Done.")

**Note on the loss floor.** The loss plateaus around ~1.7, not 0. This is because the target $x_1 - x_0$ has irreducible variance: even an *optimal* model that predicts $\mathbb{E}[x_1 - x_0 \mid x_t, t]$ cannot drive the per-sample squared error to zero. The loss is therefore a relative diagnostic, not an absolute one.

### 1.4 Sampling: integrating the ODE

In [ ]:
@torch.no_grad()
def sample_ode(model, n: int = 2000, K: int = 100) -> torch.Tensor:
    """Deterministic Euler integration of dx/dt = v(x, t) from t=0 (noise) to t=1 (data)."""
    x = torch.randn(n, 2, device=device)
    dt = 1.0 / K
    for k in range(K):
        t = torch.full((n,), k * dt, device=device)
        v = model(x, t)
        x = x + v * dt
    return x.cpu()


samples = sample_ode(base_model, n=4000, K=100)
real = sample_checkerboard(4000)

fig, axes = plt.subplots(1, 2, figsize=(10, 5))
axes[0].scatter(real[:, 0], real[:, 1], s=2, alpha=0.5)
axes[0].set_title("Real checkerboard")
axes[1].scatter(samples[:, 0], samples[:, 1], s=2, alpha=0.5)
axes[1].set_title("Flow matching samples")
for ax in axes:
    ax.set_xlim(-2.5, 2.5); ax.set_ylim(-2.5, 2.5); ax.set_aspect("equal")
plt.tight_layout(); plt.show()

**You should see** 8 visible modes in the right plot, roughly arranged in a checkerboard. The boundaries are softer than the true distribution — that is expected from a 4-layer MLP trained for 4k steps. For a teaching notebook this is fine; the point of the next sections is *not* density estimation quality but the RL fine-tuning that comes next.

---

## 2. From flow to policy: the deterministic-to-stochastic move

Right now our flow is **deterministic**: given the same $x_0$ and the same network, you always get the same $x_1$. There is nothing to "explore." To do RL, we need a *stochastic* policy that we can sample from and compute log-probs for.

### The fix: replace the ODE with an SDE

We discretize $[0, 1]$ into $K$ steps with $\Delta t = 1/K$. Instead of the deterministic update

$$x_{k+1} = x_k + v_\theta(x_k, t_k)\, \Delta t$$

we sample

$$x_{k+1} \sim \mathcal{N}\!\left(x_k + v_\theta(x_k, t_k)\, \Delta t,\; \sigma^2 \Delta t \cdot I\right)$$

This is **Euler-Maruyama integration** of the SDE

$$dx = v_\theta(x, t)\, dt + \sigma\, dW$$

In the limit $\sigma \to 0$ we recover the original deterministic flow; for $\sigma > 0$ we have a genuine stochastic policy whose log-probability under the model is tractable (it's a product of Gaussians).

### Why this works (briefly)

This SDE has the same marginal distribution at $t = 1$ as the ODE if the velocity field is appropriately corrected. For small $\sigma$ and large $K$ the correction is negligible and we can treat the SDE samples as draws from (approximately) the same distribution. We use $\sigma = 0.1$ throughout, which gives a stochastic policy without significantly distorting the prior.

### What this means in RL terms

- **Trajectory**: $\tau = (x_0, x_1, \ldots, x_K)$.
- **Policy log-prob**: $\log p_\theta(\tau) = \sum_{k=0}^{K-1} \log \mathcal{N}(x_{k+1};\, x_k + v_\theta \Delta t,\, \sigma^2 \Delta t I)$
- **Reward**: a scalar $r(x_K)$ on the final sample only (sparse, terminal).


In [ ]:
SIGMA = 0.1     # noise level for the SDE
K_STEPS = 50    # integration steps


def rollout(model, n: int, K: int = K_STEPS, sigma: float = SIGMA):
    """Roll out n stochastic trajectories. Returns (x_final, log_prob, trajectory)."""
    dt = 1.0 / K
    std = sigma * math.sqrt(dt)
    x = torch.randn(n, 2, device=device)
    log_prob = torch.zeros(n, device=device)
    traj = [x.clone()]
    for k in range(K):
        t = torch.full((n,), k * dt, device=device)
        v = model(x, t)
        mean = x + v * dt
        x_next = mean + std * torch.randn_like(x)
        # log-prob of a 2D isotropic Gaussian
        lp = -0.5 * ((x_next - mean) ** 2).sum(-1) / (std ** 2) \
             - 2 * math.log(std * math.sqrt(2 * math.pi))
        log_prob = log_prob + lp
        x = x_next
        traj.append(x.clone())
    return x, log_prob, traj


def logprob_of_trajectory(model, traj, K: int = K_STEPS, sigma: float = SIGMA):
    """Recompute log-prob of an *existing* trajectory under `model`.
    Used for the importance-sampling ratio and the reference KL.
    """
    dt = 1.0 / K
    std = sigma * math.sqrt(dt)
    log_prob = torch.zeros(traj[0].shape[0], device=device)
    for k in range(K):
        t = torch.full((traj[k].shape[0],), k * dt, device=device)
        v = model(traj[k], t)
        mean = traj[k] + v * dt
        lp = -0.5 * ((traj[k + 1] - mean) ** 2).sum(-1) / (std ** 2) \
             - 2 * math.log(std * math.sqrt(2 * math.pi))
        log_prob = log_prob + lp
    return log_prob


def kl_to_ref_gaussian(model, ref_model, traj, K: int = K_STEPS, sigma: float = SIGMA):
    """Closed-form KL(pi_theta || pi_ref) summed over the K Euler steps.

    Each per-step policy is an isotropic Gaussian
        pi(X_{t+h} | X_t) = N(X_t + h * v(X_t, t),  sigma^2 * h * I)
    with the *same* covariance under both theta and ref. So the per-step KL
    between two such Gaussians collapses to a scaled squared distance between
    the means:
        KL_step = (h / (2 * sigma^2)) * || v_theta(X_t, t) - v_ref(X_t, t) ||^2.

    Returns a tensor of shape (n,) with the trajectory-level KL.
    """
    dt = 1.0 / K
    kl = torch.zeros(traj[0].shape[0], device=device)
    for k in range(K):
        t = torch.full((traj[k].shape[0],), k * dt, device=device)
        v_theta = model(traj[k], t)
        with torch.no_grad():
            v_ref = ref_model(traj[k], t)
        kl = kl + 0.5 * dt / (sigma ** 2) * ((v_theta - v_ref) ** 2).sum(-1)
    return kl


Quick sanity check: stochastic samples should look similar to deterministic samples (because we picked $\sigma$ small).

In [ ]:
with torch.no_grad():
    sto, _, _ = rollout(base_model, n=4000)

fig, axes = plt.subplots(1, 2, figsize=(10, 5))
axes[0].scatter(samples[:, 0], samples[:, 1], s=2, alpha=0.5)
axes[0].set_title("Deterministic (ODE)")
axes[1].scatter(sto.cpu()[:, 0], sto.cpu()[:, 1], s=2, alpha=0.5)
axes[1].set_title(f"Stochastic (SDE, sigma={SIGMA})")
for ax in axes:
    ax.set_xlim(-2.5, 2.5); ax.set_ylim(-2.5, 2.5); ax.set_aspect("equal")
plt.tight_layout(); plt.show()

---

## 3. GRPO: steering with a black-box reward

We now have a stochastic policy $\pi_\theta$ that produces final samples $x_K$. We pick a reward function (deliberately a **black box** — it does not need to be differentiable, and we will only ever call it as a function) and fine-tune the policy to maximize expected reward.

### The reward

We will reward proximity to a target point at $(1.5, 1.5)$ — the center of one specific checker square in the top-right corner:

$$r(x) = \exp\!\left(-\frac{\|x - (1.5, 1.5)\|^2}{0.5}\right)$$

This is a perfectly differentiable function, but we are not going to use that fact — we treat it as a black box.

In [ ]:
TARGET = torch.tensor([1.5, 1.5], device=device)


def reward_fn(x_final: torch.Tensor) -> torch.Tensor:
    """Black-box reward: high near (1.5, 1.5)."""
    d2 = ((x_final - TARGET) ** 2).sum(dim=-1)
    return torch.exp(-d2 / 0.5)

### GRPO vs PPO in one paragraph

PPO learns a value function $V_\phi(s)$ as a baseline for advantage estimation. GRPO replaces this with a **group-relative baseline**:

1. For each *prompt* / condition, sample a *group* of $N$ trajectories.
2. Compute reward for each: $r_1, \ldots, r_N$.
3. Advantage of trajectory $i$ is $\hat A_i = (r_i - \mu) / \sigma_r$, where $\mu, \sigma_r$ are the group mean and std.

That's the only change. The rest is standard clipped-PPO surrogate plus a KL penalty to a frozen reference model (the original supervised flow). The KL keeps the policy from drifting arbitrarily far and is the *exact* same trick used in RLHF for language models.

In our setup the "condition" is empty (unconditional checkerboard), so all trajectories in an iteration form a single big group — or we split them into mini-groups by chance. The conditional generalization is a one-line change at the end of this notebook.

### The KL estimator

Because every Euler step's policy is an isotropic Gaussian with the *same* fixed covariance $\sigma^2 h\, I$ under both $\pi_\theta$ and $\pi_{\mathrm{ref}}$ (only the mean depends on $\theta$), the trajectory-level KL has a **closed form** — no Monte-Carlo estimator needed:

$$\mathrm{KL}(\pi_\theta \| \pi_{\mathrm{ref}}) \;=\; \sum_{k=0}^{K-1} \frac{h}{2\sigma^2}\,\bigl\| v_\theta(X_{t_k}, t_k) - v_{\mathrm{ref}}(X_{t_k}, t_k) \bigr\|^2.$$

This is what `kl_to_ref_gaussian` (defined above) computes. It is exact (not an estimator), deterministic given the trajectory, and lower-variance than the trajectory-level $k3$ estimator that we would otherwise need.

In [ ]:
def grpo_train(
    model,
    ref_model,
    iters: int = 150,
    group_size: int = 16,
    n_groups: int = 4,
    lr: float = 1e-4,
    kl_coef: float = 0.05,
    clip_ratio: float = 0.2,
    inner_epochs: int = 2,
    log_every: int = 20,
):
    """Group Relative Policy Optimization for a stochastic flow policy."""
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    history = {"reward": [], "kl": [], "pg_loss": []}

    for it in range(iters):
        n = group_size * n_groups

        # --- (1) Collect rollouts with current policy ---
        with torch.no_grad():
            x_final, old_logp, traj = rollout(model, n)
            r = reward_fn(x_final)                              # (n,)

        # --- (2) Group-relative advantages ---
        r_grp = r.view(n_groups, group_size)
        adv = (r_grp - r_grp.mean(1, keepdim=True)) / (r_grp.std(1, keepdim=True) + 1e-6)
        adv = adv.reshape(n).detach()

        traj_det = [s.detach() for s in traj]
        old_logp = old_logp.detach()

        # --- (3) PPO-style updates with KL penalty ---
        for _ in range(inner_epochs):
            new_logp = logprob_of_trajectory(model, traj_det)
            ratio = torch.exp(new_logp - old_logp)
            unclipped = ratio * adv
            clipped   = torch.clamp(ratio, 1 - clip_ratio, 1 + clip_ratio) * adv
            pg_loss = -torch.min(unclipped, clipped).mean()

            # Closed-form Gaussian KL: per-step Gaussians with identical covariance
            # under theta and ref => KL collapses to a scaled L2 distance of means.
            kl = kl_to_ref_gaussian(model, ref_model, traj_det).mean()

            loss = pg_loss + kl_coef * kl
            opt.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()

        history["reward"].append(r.mean().item())
        history["kl"].append(kl.item())
        history["pg_loss"].append(pg_loss.item())

        if it % log_every == 0:
            print(f"  iter {it:3d}: reward {r.mean():.3f}  KL {kl:.3f}  pg_loss {pg_loss:.4f}")
    return history

### 3.1 Running GRPO

In [ ]:
# Frozen reference model = a snapshot of the original flow.
# The KL penalty pulls the policy toward this distribution.
ref_model = copy.deepcopy(base_model)
for p in ref_model.parameters():
    p.requires_grad_(False)

# Make a copy to fine-tune (so we can keep the base for comparison plots)
policy = copy.deepcopy(base_model)

print("Running GRPO...")
history = grpo_train(policy, ref_model, iters=150, kl_coef=0.05)
print("Done.")

### 3.2 Before vs. after

In [ ]:
with torch.no_grad():
    after, _, _ = rollout(policy, n=4000)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
axes[0].scatter(samples[:, 0], samples[:, 1], s=2, alpha=0.5)
axes[0].scatter([1.5], [1.5], c="red", s=300, marker="*", zorder=5, edgecolors="black", linewidths=1)
axes[0].set_title("Before RL (base flow)")

axes[1].scatter(after.cpu()[:, 0], after.cpu()[:, 1], s=2, alpha=0.5, color="C2")
axes[1].scatter([1.5], [1.5], c="red", s=300, marker="*", zorder=5, edgecolors="black", linewidths=1)
axes[1].set_title("After GRPO (steered toward target)")

for ax in axes[:2]:
    ax.set_xlim(-2.5, 2.5); ax.set_ylim(-2.5, 2.5); ax.set_aspect("equal")

axes[2].plot(history["reward"])
axes[2].set_xlabel("iteration"); axes[2].set_ylabel("mean reward")
axes[2].set_title("Reward over training")
axes[2].grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

**What just happened.** GRPO concentrated mass near the target without ever differentiating through the reward function. It only called `reward_fn` on samples and used the resulting scalars to weight policy gradient steps.

If you look closely, the post-GRPO distribution **still has some structure from the original checkerboard** — there is residual mass at other modes. That's the KL term keeping us close to the prior. If we crank `kl_coef` down to 0, we should see more aggressive concentration. Let's verify.

---

## 4. The KL/reward trade-off

This is the most pedagogically important section: KL regularization controls **how much we are willing to distort the prior in exchange for reward**. This is the same lever used in RLHF for language models.

In [ ]:
# WARNING: this cell does three GRPO runs. On CPU it takes ~6-8 minutes.
# On GPU it should finish in well under a minute.

kl_coefs = [0.0, 0.05, 0.5]
sweep_results = {}

for kl_coef in kl_coefs:
    print(f"\n=== kl_coef = {kl_coef} ===")
    m = copy.deepcopy(base_model)
    r = copy.deepcopy(base_model)
    for p in r.parameters():
        p.requires_grad_(False)
    h = grpo_train(m, r, iters=100, kl_coef=kl_coef, log_every=25)
    with torch.no_grad():
        s, _, _ = rollout(m, n=3000)
    sweep_results[kl_coef] = (s.cpu(), h)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for ax, kl_coef in zip(axes, kl_coefs):
    s, h = sweep_results[kl_coef]
    ax.scatter(s[:, 0], s[:, 1], s=2, alpha=0.5)
    ax.scatter([1.5], [1.5], c="red", s=200, marker="*", zorder=5, edgecolors="black", linewidths=1)
    ax.set_title(f"kl_coef = {kl_coef}\nfinal mean reward = {h['reward'][-1]:.2f}")
    ax.set_xlim(-2.5, 2.5); ax.set_ylim(-2.5, 2.5); ax.set_aspect("equal")
plt.tight_layout(); plt.show()

**Read the plots left-to-right:**

- **`kl_coef = 0` (no regularization)**: the policy is free to do whatever maximizes reward. Expect aggressive collapse onto the target — at the cost of forgetting the checkerboard prior entirely.
- **`kl_coef = 0.05` (light regularization)**: a clear shift toward the target, but the original modes are still visible. This is roughly the regime used in RLHF.
- **`kl_coef = 0.5` (heavy regularization)**: the policy barely moves. The KL term dominates; reward gain is small.

This is **the central trade-off of RL fine-tuning of generative models**. Too little KL → mode collapse, loss of diversity, "reward hacking" in language models. Too much KL → the reward signal is ignored. Choosing the right coefficient (or the right schedule) is one of the main practical art-forms in modern RLHF.

---

## 5. Bonus: conditional generation (and why GRPO is designed this way)

So far we have a single reward function and no notion of a "prompt." In LLM RLHF, each training example has a prompt and the policy generates a *response*. GRPO is designed for that setting: it samples $N$ responses **per prompt** and computes advantages within each prompt's group.

To see this clearly in our toy domain, we make the target *conditional*: pass in a one-hot $c \in \{\text{TL}, \text{TR}, \text{BL}, \text{BR}\}$ that specifies which corner to land in. Then GRPO samples $N$ trajectories per corner, and the group baseline is naturally per-corner.

This subsection is left as an **exercise**. The hooks below give you the pieces:

1. Modify `VelocityNet` to accept a 4-dim one-hot `c` (we already left the hook for this in `cond_dim`).
2. Pre-train it on `(x_1, c)` pairs where $c$ is a one-hot indicating which corner $x_1$ belongs to (sample only from that corner of the checkerboard given $c$).
3. Modify `rollout` to accept a batch of conditions, and `reward_fn` to depend on the condition (high reward = land in the corner specified by $c$).
4. In `grpo_train`, organize groups *by condition*: each group of $N$ trajectories shares the same $c$. Group-normalize rewards within each group.

If you do this, you will reproduce the *actual* RLHF setup: a conditional generative model fine-tuned with per-prompt group-relative advantages.

---

## What to take away

1. **Flow matching** is a simple, clean way to train generative models: regress a velocity field along straight-line paths between noise and data.
2. **Turning a generative model into an RL policy** requires only one structural change: replace the deterministic ODE with a stochastic SDE so trajectories have tractable log-probs.
3. **GRPO** is "PPO without a value function": replace the learned baseline with a group-relative one. Especially powerful when there is a natural notion of "prompt" giving each sample a small group of siblings.
4. **KL regularization** controls the trade-off between reward maximization and staying close to the prior. This is the central knob in modern RLHF.
5. The whole picture — generative prior + RL fine-tuning + KL-to-reference — is the **shared template** behind RLHF for LLMs, diffusion/flow RL for images and video, and modern robotics policies.

### Suggested exercises

- **(easy)** Move the target to $(-1.5, -1.5)$ and re-run. Does GRPO find it equally well from any starting prior mode?
- **(medium)** Replace the smooth reward with a sparse one: $r(x) = 1$ if $x$ is inside the top-right unit square, else 0. How does training change? (Hint: advantages become more degenerate.)
- **(medium)** Plot the *KL divergence to the reference* as a function of training iteration for each `kl_coef`. Confirm that low `kl_coef` → high KL, and vice versa.
- **(hard)** Implement the conditional version in Section 5 and verify that GRPO learns to route to the right corner given $c$.
- **(hard)** Replace GRPO with a *differentiable* objective: since `reward_fn` is differentiable, you can backprop directly through the SDE. Compare sample efficiency. When would you prefer RL? (Hint: think about non-differentiable reward models.)
